### RAG Pipelines - Data ingestion to Vector DB Pipeline

In [ ]:
import os
from pathlib import Path

from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


C:\Users\hp_pa\AppData\Local\Temp\ipykernel_13240\3230511120.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader


In [ ]:
### Read all the PDF's inside the directory

def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"

            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")

        except Exception as e:
            print(f"  ✕ Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents


# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files to process

Processing: Classification Model Evaluation_ Case Study Solutions.pdf
  ✓ Loaded 1 pages

Processing: Regression Diagnostics_ Patient Length of Stay.pdf
  ✓ Loaded 1 pages

Total documents loaded: 2


In [5]:
all_pdf_documents

[Document(metadata={'producer': 'Skia/PDF m136', 'creator': 'Chromium', 'creationdate': '2026-10-14T18:48:49+00:00', 'title': 'Classification Model Diagnostics: Case Study Solutions', 'moddate': '2026-10-14T18:48:49+00:00', 'source': '..\\data\\pdf\\Classification Model Evaluation_ Case Study Solutions.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Classification Model Evaluation_ Case Study Solutions.pdf', 'file_type': 'pdf'}, page_content='DIAGNOSTIC MATRIX 1 · CLASSIFICATION SOLUTIONS\nClassification Model Evaluation: Case Study Solutions\nBenchmarked Case Study: An automated email ﬁltering model was evaluated on a veriﬁed test set of 100 emails.\nThe goal is separating Spam (Positive Class) from Legitimate Mail / Ham (Negative Class) with minimal false\nalarms and minimal leaks.\n1. The Confusion Matrix (N = 100)\nGround Truth\nPredicted Condition\nActual Totals\nPredicted Spam (Positive) Predicted Ham (Negative)\nActual Spam (Positive) True Positive (TP) = 4

In [8]:
### Text splitting get into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"split: {len(documents)} documents into {len(split_docs)} chunks")
    
    # show example of a chunk
    if split_docs:
        print(f"Example chunk content:\n{split_docs[0].page_content[:200]}...")  # Show first 500 characters of the first chunk
        print(f"Example chunk metadata:\n{split_docs[0].metadata}")
        
    return split_docs

In [10]:
chuned_documents = split_documents(all_pdf_documents)
chuned_documents

split: 2 documents into 6 chunks
Example chunk content:
DIAGNOSTIC MATRIX 1 · CLASSIFICATION SOLUTIONS
Classification Model Evaluation: Case Study Solutions
Benchmarked Case Study: An automated email ﬁltering model was evaluated on a veriﬁed test set of 10...
Example chunk metadata:
{'producer': 'Skia/PDF m136', 'creator': 'Chromium', 'creationdate': '2026-10-14T18:48:49+00:00', 'title': 'Classification Model Diagnostics: Case Study Solutions', 'moddate': '2026-10-14T18:48:49+00:00', 'source': '..\\data\\pdf\\Classification Model Evaluation_ Case Study Solutions.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Classification Model Evaluation_ Case Study Solutions.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Skia/PDF m136', 'creator': 'Chromium', 'creationdate': '2026-10-14T18:48:49+00:00', 'title': 'Classification Model Diagnostics: Case Study Solutions', 'moddate': '2026-10-14T18:48:49+00:00', 'source': '..\\data\\pdf\\Classification Model Evaluation_ Case Study Solutions.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Classification Model Evaluation_ Case Study Solutions.pdf', 'file_type': 'pdf'}, page_content='DIAGNOSTIC MATRIX 1 · CLASSIFICATION SOLUTIONS\nClassification Model Evaluation: Case Study Solutions\nBenchmarked Case Study: An automated email ﬁltering model was evaluated on a veriﬁed test set of 100 emails.\nThe goal is separating Spam (Positive Class) from Legitimate Mail / Ham (Negative Class) with minimal false\nalarms and minimal leaks.\n1. The Confusion Matrix (N = 100)\nGround Truth\nPredicted Condition\nActual Totals\nPredicted Spam (Positive) Predicted Ham (Negative)\nActual Spam (Positive) True Positive (TP) = 4

### Embeding And VectorStoreDB 

In [19]:
import uuid
from typing import Any, Dict, List, Tuple

import chromadb
import numpy as np
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


ModuleNotFoundError: No module named 'chromadb'

In [ ]:
class EmbeddingManager:
    """Handle document embeddings generation using Sentence Transformers."""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the EmbeddingManager 
        
        Args:
            model_name: HuggingFace model name for  sentence embeddings.
        """
        
        self.model_name = model_name
        self.model = None
        self._load_model()
        
    def _load_model(self):
        """Load the Sentence Transformer model."""
        try:
            print(f"Loading embedding model: {self.model_name}...")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
        
    def generate_embeddings(self, documents: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of documents.
        
        Args:
            documents: List of document strings to embed.
        """
        if not self.model:
            raise ValueError("Model is not loaded. Call _load_model() first.")
        
        try:
            embeddings = self.model.encode(documents, convert_to_numpy=True)
            print(f"Generated embeddings for {len(documents)} documents.")
            print(f"Generated Embeddings with shape: {embeddings.shape}")
            return embeddings
        except Exception as e:
            print(f"Error generating embeddings: {e}")
            raise
        
## initialize the EmbeddingManager and generate embeddings for the chunks

embedding_manager = EmbeddingManager()
embedding_manager

### Vector Store